# Step 09: µP Learning Rate Sweep (Tiny Model)
Upload `train.npy` and `val.npy` to Google Drive at `MyDrive/ML_project/data/splits/`

In [3]:
!pip install -q mup
import os, sys, json, time, math, shutil
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F
from mup import MuReadout, MuSharedReadout, set_base_shapes
from mup.optim import MuAdamW

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Setup paths
BASE = Path("/content/ML_project")
BASE.mkdir(exist_ok=True)
DATA = BASE / "data" / "splits"
DATA.mkdir(parents=True, exist_ok=True)
CKPT = BASE / "checkpoints" / "mup_lr_sweep"
CKPT.mkdir(parents=True, exist_ok=True)

# Copy data from Drive (adjust path if needed)
DRIVE_DATA = Path("/content/drive/MyDrive/ML_project/data/splits")
for f in ["train.npy", "val.npy"]:
    src = DRIVE_DATA / f
    dst = DATA / f
    if src.exists() and not dst.exists():
        print(f"Copying {f}...")
        shutil.copy2(src, dst)
    elif dst.exists():
        print(f"{f} already present")
    else:
        raise FileNotFoundError(f"Upload {f} to Google Drive at {DRIVE_DATA}")

print(f"Device: {torch.cuda.get_device_name(0)}")
# print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

KeyboardInterrupt: 

In [4]:
@dataclass
class GPTConfig:
    block_size: int = 2048
    vocab_size: int = 4096
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    d_ff: int = None
    dropout: float = 0.0
    bias: bool = False

MODEL_CONFIGS = {
    "tiny": GPTConfig(n_layer=4, n_head=4, n_embd=128, d_ff=512),
    "small": GPTConfig(n_layer=6, n_head=6, n_embd=192, d_ff=768),
    "medium": GPTConfig(n_layer=6, n_head=6, n_embd=384, d_ff=1536),
    "large": GPTConfig(n_layer=10, n_head=8, n_embd=512, d_ff=2048),
    "xl": GPTConfig(n_layer=12, n_head=12, n_embd=768, d_ff=3072),
}

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class MuCausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.dropout = config.dropout
        self.q_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.k_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.v_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
            dropout_p=self.dropout if self.training else 0,
            is_causal=True, scale=8.0/self.head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MuMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        d_ff = config.d_ff or 4 * config.n_embd
        self.c_fc = nn.Linear(config.n_embd, d_ff, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(d_ff, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class MuBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = MuCausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MuMLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))

class MuGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([MuBlock(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = MuSharedReadout(self.transformer.wte.weight, bias=False)

    def _init_weights_mup(self):
        for name, module in self.named_modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=1.0/math.sqrt(module.weight.shape[1]))
                if module.bias is not None: nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
        for block in self.transformer.h:
            nn.init.zeros_(block.attn.q_proj.weight)
            if block.attn.q_proj.bias is not None: nn.init.zeros_(block.attn.q_proj.bias)
        for block in self.transformer.h:
            for proj in [block.attn.c_proj, block.mlp.c_proj]:
                std = 1.0/math.sqrt(proj.weight.shape[1]) / math.sqrt(2*self.config.n_layer)
                nn.init.normal_(proj.weight, mean=0.0, std=std)

    def get_num_params(self, non_embedding=True):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding: n -= self.transformer.wpe.weight.numel()
        return n

    def forward(self, idx, targets=None):
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h: x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :]); loss = None
        return logits, loss

def create_mup_model(model_name):
    cfg = MODEL_CONFIGS[model_name]
    n_head = cfg.n_head
    bw, dw = n_head, n_head * 2
    base_cfg = GPTConfig(n_layer=cfg.n_layer, n_head=n_head, n_embd=bw, d_ff=bw*4,
                         block_size=cfg.block_size, vocab_size=cfg.vocab_size, bias=cfg.bias)
    delta_cfg = GPTConfig(n_layer=cfg.n_layer, n_head=n_head, n_embd=dw, d_ff=dw*4,
                          block_size=cfg.block_size, vocab_size=cfg.vocab_size, bias=cfg.bias)
    base_m, delta_m, model = MuGPT(base_cfg), MuGPT(delta_cfg), MuGPT(cfg)
    set_base_shapes(model, base_m, delta=delta_m)
    model._init_weights_mup()
    del base_m, delta_m
    return model

print("Model code loaded.")

Model code loaded.


In [5]:
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def evaluate(model, data, block_size, batch_size, eval_batches, device, ctx):
    model.eval()
    losses = [0.0]*eval_batches
    for i in range(eval_batches):
        x, y = get_batch(data, block_size, batch_size, device)
        with ctx: _, loss = model(x, y)
        losses[i] = loss.item()
    model.train()
    return float(np.mean(losses))

def get_lr(step, warmup, max_steps, max_lr, min_lr):
    if step < warmup: return max_lr * (step+1) / warmup
    if step >= max_steps: return min_lr
    r = (step - warmup) / (max_steps - warmup)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * r)) * (max_lr - min_lr)

def train_mup(model_name, lr, micro_bs, grad_accum, train_path, val_path,
              warmup=100, save_ckpt=False, ckpt_dir=None):
    device = "cuda"
    ctx = torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)
    train_data = np.load(train_path, mmap_mode='r')
    val_data = np.load(val_path, mmap_mode='r')
    block_size = 2048
    tokens_per_step = micro_bs * grad_accum * block_size
    total_steps = len(train_data) // tokens_per_step

    model = create_mup_model(model_name).to(device)
    n_params = model.get_num_params()
    print(f"  Model '{model_name}': {n_params:,} params ({n_params/1e6:.2f}M), LR={lr}")

    pd = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
    decay = [p for n, p in pd.items() if p.dim() >= 2]
    nodecay = [p for n, p in pd.items() if p.dim() < 2]
    optimizer = MuAdamW([{'params': decay, 'weight_decay': 0.1},
                         {'params': nodecay, 'weight_decay': 0.0}], lr=lr, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler(enabled=False)
    min_lr = lr * 0.1
    init_lrs = [pg['lr'] for pg in optimizer.param_groups]

    model.train(); t0 = time.time(); total_tok = 0; best_vl = float('inf')
    train_losses, val_losses = [], []

    for step in range(total_steps):
        base_lr = get_lr(step, warmup, total_steps, lr, min_lr)
        ratio = base_lr / lr
        for pg, ilr in zip(optimizer.param_groups, init_lrs): pg['lr'] = ilr * ratio
        optimizer.zero_grad(set_to_none=True)
        accum_loss = 0.0
        for _ in range(grad_accum):
            x, y = get_batch(train_data, block_size, micro_bs, device)
            with ctx: _, loss = model(x, y); loss = loss / grad_accum
            scaler.scale(loss).backward(); accum_loss += loss.item(); total_tok += x.numel()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        if (step+1) % 10 == 0: train_losses.append((step, accum_loss))
        if (step+1) % 50 == 0 or step == total_steps-1:
            vl = evaluate(model, val_data, block_size, micro_bs, 20, device, ctx)
            val_losses.append((step, vl))
            if vl < best_vl: best_vl = vl
            print(f"    step {step+1}/{total_steps} | lr={base_lr:.6f} | val_loss={vl:.4f}")

    wall = time.time() - t0
    tps = total_tok / wall if wall > 0 else 0
    final_tl = train_losses[-1][1] if train_losses else 0
    final_vl = val_losses[-1][1] if val_losses else 0
    print(f"  Done in {wall:.0f}s | best_val={best_vl:.4f} | tok/s={tps:,.0f}")

    result = {"model_name": model_name, "n_params": n_params, "learning_rate": lr,
              "best_val_loss": best_vl, "final_val_loss": final_vl, "final_train_loss": final_tl,
              "wall_time_seconds": wall, "tokens_per_second": tps, "parameterization": "mup",
              "train_losses": train_losses, "val_losses": val_losses,
              "config": {"n_layer": MODEL_CONFIGS[model_name].n_layer,
                         "n_head": MODEL_CONFIGS[model_name].n_head,
                         "n_embd": MODEL_CONFIGS[model_name].n_embd,
                         "d_ff": MODEL_CONFIGS[model_name].d_ff or 4*MODEL_CONFIGS[model_name].n_embd},
              "peak_gpu_memory_mb": torch.cuda.max_memory_allocated()/1e6}

    if save_ckpt and ckpt_dir:
        cd = Path(ckpt_dir) / model_name; cd.mkdir(parents=True, exist_ok=True)
        with open(cd/"metrics.json", 'w') as f: json.dump(result, f, indent=2, default=str)
        torch.save({'model_state_dict': model.state_dict(), 'metrics': result}, cd/"model.pt")

    del model, optimizer; torch.cuda.empty_cache()
    return result

print("Training utilities loaded.")

Training utilities loaded.


In [6]:
TRAIN_PATH = str(DATA / "train.npy")
VAL_PATH = str(DATA / "val.npy")

# A100 can handle larger batches — micro_batch=64, grad_accum=4
LR_VALUES = [3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1]
results = []

for i, lr in enumerate(LR_VALUES):
    print(f"\n[{i+1}/{len(LR_VALUES)}] LR = {lr}")
    try:
        r = train_mup("tiny", lr, micro_bs=64, grad_accum=4,
                      train_path=TRAIN_PATH, val_path=VAL_PATH, warmup=100)
        results.append({"lr": lr, "best_val_loss": r["best_val_loss"],
                        "final_train_loss": r["final_train_loss"],
                        "wall_time": r["wall_time_seconds"]})
    except Exception as e:
        print(f"  FAILED: {e}")
        results.append({"lr": lr, "best_val_loss": float('inf'),
                        "final_train_loss": float('inf'), "wall_time": 0})


[1/7] LR = 0.0003
  Model 'tiny': 1,311,872 params (1.31M), LR=0.0003
    step 50/281 | lr=0.000150 | val_loss=8.2670
    step 100/281 | lr=0.000300 | val_loss=8.1725
    step 150/281 | lr=0.000254 | val_loss=8.0693
    step 200/281 | lr=0.000145 | val_loss=7.9917
    step 250/281 | lr=0.000050 | val_loss=7.9532
    step 281/281 | lr=0.000030 | val_loss=7.9464
  Done in 62s | best_val=7.9464 | tok/s=2,367,219

[2/7] LR = 0.001
  Model 'tiny': 1,311,872 params (1.31M), LR=0.001
    step 50/281 | lr=0.000500 | val_loss=8.1848
    step 100/281 | lr=0.001000 | val_loss=7.9076
    step 150/281 | lr=0.000847 | val_loss=7.5108
    step 200/281 | lr=0.000484 | val_loss=7.2108
    step 250/281 | lr=0.000168 | val_loss=7.0738
    step 281/281 | lr=0.000100 | val_loss=7.0397
  Done in 61s | best_val=7.0397 | tok/s=2,407,006

[3/7] LR = 0.003
  Model 'tiny': 1,311,872 params (1.31M), LR=0.003
    step 50/281 | lr=0.001500 | val_loss=8.0290
    step 100/281 | lr=0.003000 | val_loss=7.0610
    step

In [7]:
valid = [r for r in results if r["best_val_loss"] < float('inf')]
best = min(valid, key=lambda r: r["best_val_loss"])
best_lr = best["lr"]

print(f"\n{'='*60}\nmuP LR SWEEP RESULTS\n{'='*60}")
print(f"{'LR':>10} | {'Val Loss':>10} | {'Train Loss':>10} | {'Time':>8}")
print("-"*50)
for r in results:
    m = " <-- BEST" if r["lr"] == best_lr else ""
    vl = f"{r['best_val_loss']:.4f}" if r['best_val_loss'] < float('inf') else "FAILED"
    print(f"{r['lr']:>10.1e} | {vl:>10} | {r['final_train_loss']:.4f} | {r['wall_time']:>7.0f}s{m}")
print(f"\nBest muP LR: {best_lr}")

# Save
with open(CKPT / "sweep_results.json", 'w') as f:
    json.dump({"results": results, "best_lr": best_lr}, f, indent=2)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
lrs = [r["lr"] for r in valid]; vls = [r["best_val_loss"] for r in valid]
ax.semilogx(lrs, vls, 'o-', color='#E53935', markersize=8, linewidth=2, label='muP')
ax.axvline(best_lr, color='#4CAF50', ls='--', alpha=0.7, label=f'Best: {best_lr:.1e}')
ax.set_xlabel("Learning Rate"); ax.set_ylabel("Best Val Loss")
ax.set_title("muP LR Sweep (Tiny, 1 Epoch)"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CKPT / "mup_lr_sweep_plot.png", dpi=150); plt.show(); plt.close()

# Copy results to Drive
DRIVE_CKPT = Path("/content/drive/MyDrive/ML_project/checkpoints/mup_lr_sweep")
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
shutil.copy2(CKPT/"sweep_results.json", DRIVE_CKPT/"sweep_results.json")
shutil.copy2(CKPT/"mup_lr_sweep_plot.png", DRIVE_CKPT/"mup_lr_sweep_plot.png")
print(f"Results saved to Drive: {DRIVE_CKPT}")


muP LR SWEEP RESULTS
        LR |   Val Loss | Train Loss |     Time
--------------------------------------------------
   3.0e-04 |     7.9464 | 7.9471 |      62s
   1.0e-03 |     7.0397 | 7.0396 |      61s
   3.0e-03 |     4.6036 | 4.6078 |      61s
   1.0e-02 |     2.2239 | 2.2332 |      61s
   3.0e-02 |     1.9668 | 2.0077 |      61s
   1.0e-01 |     1.6721 | 1.6831 |      61s <-- BEST
   3.0e-01 |     1.8047 | 1.8046 |      61s

Best muP LR: 0.1
Results saved to Drive: /content/drive/MyDrive/ML_project/checkpoints/mup_lr_sweep
